# Notebook C — Full Evaluation, Physics Validation, Cross-Sim, Plots

## Setup (carried over from your already-run Notebook A + Notebook B)
Same 6 setup cells as Notebook A, plus Notebook B's embeddings/helper cells — but
**not** B's training loop. Instead of retraining the policy a second time, this
loads the `policy.pt` your Notebook B run already saved.

**Fix applied here that wasn't in the original spec:** Notebook B's "verify" cell
(the one that computes `preds_pol` / `preds_full` / `targets` — needed by this
notebook's final results table) was never listed as a prerequisite for this
notebook, even though the results-table cell depends on it. It's included below
labeled **CELL B-VERIFY**.

## Setup (from your already-run Notebook A)

In [ ]:
import torch, sys, subprocess
print("torch:", torch.__version__, "| python:", sys.version.split()[0], "| cuda:", torch.cuda.is_available())
TORCH = torch.__version__.split("+")[0]
def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(args))
pip("torch-geometric==2.6.1", "torch-scatter", "torch-sparse", "torch-cluster",
    "-f", f"https://data.pyg.org/whl/torch-{TORCH}+cpu.html")
pip("pyyaml", "scipy", "pandas", "matplotlib", "seaborn")
print("deps OK")

In [ ]:
import os
import subprocess

REPO_DIR = "/kaggle/working/cosmic-net"
# Audited implementation lives on fork fix/rl-pruning-symmetry (upstream main
# lags the audited commits); pin both so execution matches the reviewed code.
REPO_URL = "https://github.com/Neal-Salian/cosmic-net-f.git"
REPO_BRANCH = "fix/rl-pruning-symmetry"

# Read GitHub PAT from Kaggle Secrets
# If using kaggle_secrets:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
GITHUB_PAT = user_secrets.get_secret("GITHUB_PAT")

# Clone only if repo doesn't already exist
if not os.path.exists(REPO_DIR):
    auth_url = REPO_URL.replace(
        "https://",
        f"https://x-access-token:{GITHUB_PAT}@"
    )

    # SECURITY: never let the token reach the notebook output. A failed
    # subprocess.check_call raises CalledProcessError whose message embeds the
    # full command line (i.e. the token) — re-raise a sanitized error instead.
    try:
        subprocess.check_call([
            "git", "clone",
            "--branch", REPO_BRANCH,
            auth_url,
            REPO_DIR
        ])
    except subprocess.CalledProcessError:
        raise RuntimeError(
            f"git clone failed for {REPO_URL} (token redacted) — check the PAT "
            "secret, the repo URL, and Kaggle internet access."
        ) from None
    # The clone writes the token into .git/config (remote origin URL) — remove it.
    subprocess.check_call(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])

# FIX (Sep 2026 rewire): the audited RL fixes live on fix/rl-pruning-symmetry.
# A cached REPO_DIR from an older Kaggle run would otherwise silently execute
# stale pre-fix code — always fetch + check out the branch, then scrub the PAT
# from the remote URL again (fetch writes it back into .git/config).
auth_fetch = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_PAT}@")
try:
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", auth_fetch, REPO_BRANCH])
except subprocess.CalledProcessError:
    raise RuntimeError(f"git fetch failed for {REPO_URL} (token redacted) — check the PAT secret and Kaggle internet access.") from None
subprocess.check_call(["git", "-C", REPO_DIR, "checkout", "-B", REPO_BRANCH, "FETCH_HEAD"])
subprocess.check_call(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])
print("branch:", subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"], text=True).strip(),
      "| HEAD:", subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], text=True).strip())


os.chdir(REPO_DIR)
print(os.listdir("."))

In [ ]:
# Mount uploaded data (repo already cloned privately in the previous cell)
import subprocess, os, shutil
os.chdir("/kaggle/working/cosmic-net")
print(os.listdir("."))
# Upload tng100_clustered.csv + best_model_augmented.pt as a Kaggle dataset named "cosmicnet-data"
INPUT = "/kaggle/input/datasets/nealsalian/cosmicnet-data"
os.makedirs("data/raw", exist_ok=True)
shutil.copy(f"{INPUT}/tng100_clustered.csv", "data/raw/tng100_clustered.csv")
os.makedirs("kaggle", exist_ok=True)
if os.path.exists(f"{INPUT}/best_model_augmented.pt"):
    shutil.copy(f"{INPUT}/best_model_augmented.pt", "kaggle/best_model_augmented.pt")
print("data staged:", os.path.getsize("data/raw/tng100_clustered.csv")/1e6, "MB")


In [ ]:
#Load config, force CPU-safe worker settings
import sys, yaml, torch, numpy as np
sys.path.insert(0, ".")
with open("config/config.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["data"]["source"] = "tng"
cfg["data"]["num_workers"] = 0         
cfg["data"]["batch_size"] = 16
cfg["model"]["mc_samples"] = 30
cfg["rls"] = {
    "policy_hidden": 64, "lr": 0.001, "entropy_coef": 0.01, "value_coef": 0.5,
    "epochs": 60, "batch_size": 32,
    "target_sparsity_start": 0.9, "target_sparsity_end": 0.4,
    "sparsity_anneal_epochs": 40,
    "w_acc": 1.0, "w_sp": 0.5, "w_conn": 1.0, "w_virial": 1.0, "w_unc": 0.5,
    "virial_anneal_start_epoch": 10, "min_keep_frac": 0.1, "seed": 42,
    # TTA ("RL at inference") — tune on VAL split only, then freeze
    "tta_lr": 1e-4, "tta_steps": 10, "tta_mc_samples": 15, "tta_patience": 3,
    "tta_target_sparsity": 0.5,
}

# Sep 2026 fixes (wired to rls/ below): guarded Stage B keys. sparsity_mode is
# NOT defaulted here on purpose — CELL 10-LOAD syncs it from Notebook B's
# recorded config so eval decoding matches the training run.
cfg["rls"].setdefault("stageb_epochs", 10)
cfg["rls"].setdefault("stageb_lr", 1e-4)
cfg["rls"].setdefault("stageb_patience", 3)
cfg["rls"].setdefault("stageb_full_tol", 0.02)
cfg["rls"].setdefault("stageb_augment", "policy")
cfg["rls"].setdefault("tta_val_tol", 0.02)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| source:", cfg["data"]["source"])

In [ ]:
# PROVENANCE (recording only - no computation is affected; no secrets are read)
import hashlib, json, platform, subprocess, time

def _md5_of(path, _blk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(_blk), b""):
            h.update(chunk)
    return h.hexdigest()

def _file_info(path):
    import os
    ex = os.path.exists(path)
    return {"path": path, "exists": ex,
            "size_bytes": os.path.getsize(path) if ex else None,
            "md5": _md5_of(path) if ex else None}

def _git_info(*args):
    try:
        return subprocess.check_output(["git", "-C", REPO_DIR, *args],
                                       text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        return None

def record_provenance(stage, extra=None):
    """Append one stage entry to outputs/rls/provenance.json (keyed by stage,
    so A/B/B_policy/C/D entries coexist and are never silently overwritten)."""
    import os
    entry = {
        "stage": stage,
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "repo": {"head": _git_info("rev-parse", "HEAD"),
                 "branch": _git_info("rev-parse", "--abbrev-ref", "HEAD"),
                 "describe": _git_info("describe", "--always")},
        "dataset": {"tng_csv": _file_info("data/raw/tng100_clustered.csv"),
                    "checkpoint": _file_info(f"{INPUT}/best_model_augmented.pt")},
        "seed": cfg.get("seed"),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda": {"available": torch.cuda.is_available(),
                 "version": str(torch.version.cuda),
                 "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},
        "config_used": {"data": cfg.get("data"), "graph": cfg.get("graph"),
                        "model": cfg.get("model"), "rls": cfg.get("rls")},
    }
    if extra:
        entry.update(extra)
    os.makedirs("outputs/rls", exist_ok=True)
    prov = {}
    try:
        with open("outputs/rls/provenance.json") as f:
            prov = json.load(f)
    except Exception:
        pass
    prov[stage] = entry
    with open("outputs/rls/provenance.json", "w") as f:
        json.dump(prov, f, indent=2)
    print(f"[provenance] stage '{stage}' -> outputs/rls/provenance.json "
          f"(HEAD={entry['repo']['head']}, csv md5={entry['dataset']['tng_csv']['md5']})")

record_provenance("C")

In [ ]:
#Load halos, build graphs, split
from data.loaders.base_loader import get_loader
from graph.graph_builder import GraphBuilder, build_dataloaders
loader = get_loader(cfg)
halos = loader.load()
print("total halos:", len(halos), "| split", loader.split_data.__name__ if False else "")
train_halos, val_halos, test_halos = loader.split_data()
print(f"train={len(train_halos)} val={len(val_halos)} test={len(test_halos)}")
torch.manual_seed(cfg["seed"]); np.random.seed(cfg["seed"])   # AFTER split_data (it resets RNG)
train_loader, val_loader, test_loader = build_dataloaders(cfg, train_halos, val_halos, test_halos)
b0 = next(iter(test_loader))
print("graph:", b0.x.shape, b0.edge_index.shape, b0.edge_attr.shape, "y:", b0.y.shape)

In [ ]:
# Load frozen backbone and verify it predicts
from model.model import load_model
import torch
import torch_geometric.data as pyg_data

ckpt = "/kaggle/input/datasets/nealsalian/cosmicnet-data/best_model_augmented.pt"

# ---------------------------------------------------------
# Load model
# ---------------------------------------------------------
gnn = load_model(ckpt, cfg, device)
gnn.eval()

model_device = next(gnn.parameters()).device

# ---------------------------------------------------------
# Create a valid test graph
# Use the complete graph instead of taking 2 nodes
# with arbitrary edges.
# ---------------------------------------------------------
single = pyg_data.Data(
    x=b0.x,
    edge_index=b0.edge_index,
    edge_attr=b0.edge_attr
)

# One graph containing all nodes
single.batch = torch.zeros(
    single.x.shape[0],
    dtype=torch.long
)

# Move graph to the same device as the model
single = single.to(model_device)

# ---------------------------------------------------------
# Forward pass
# ---------------------------------------------------------
with torch.no_grad():
    pred, _ = gnn(single)

print(
    "smoke prediction:", pred.item(),
    "| device:", model_device,
    "| params:", sum(p.numel() for p in gnn.parameters())
)

In [ ]:
# RL helpers — imported from the committed rls/ package (Sep 2026 rewire).
# These were previously reimplemented inline in this cell; the inline copies
# drifted from rls/ (notably: no pairwise-symmetric masking, no
# topk_scheduled mode, pre-fix TTA), so this cell is imports-only now.
# Drift guard: tests/test_notebook_hygiene.py fails CI if inline copies return.
import torch
import torch.nn as nn
import torch.nn.functional as F

from rls.policy import EdgePolicyNet, build_policy
from rls.policy_gradient import (bernoulli_logp, bernoulli_entropy,
                                 sample_actions, compute_advantages,
                                 compute_pg_loss, PolicyGradientTrainer,
                                 ValueNet)
from rls.sparsify import (hard_mask, apply_min_keep_floor, repair_connectivity,
                          symmetrize_probs, repair_symmetric,
                          final_symmetric_mask, topk_scheduled_mask, eval_mask,
                          pair_asymmetry_fraction)
from rls.train_policy import (train_policy, prepare_graphs,
                              check_curriculum_divergence,
                              _graph_physics_terms as graph_physics_terms)
from rls.rewards import (compute_rewards, relative_virial_penalty,
                         virial_ratio_pruned, label_free_reward)
from rls.stageb import fine_tune_gnn, edge_dropout_masks
from rls.tta import adapt_at_test_time, mc_std, edge_kl, tta_should_enable
from rls.evaluate import build_results_table, save_paper_plots
from rls.provenance import (record_backbone, format_backbone_label,
                            require_backbone_label)


## Precompute embeddings + RL training helpers (needed by cells below, but the training loop itself is skipped)

In [ ]:
# CELL 6: Precompute frozen node embeddings + graph contexts for ALL graphs
# (Sep 2026 rewire: uses rls.train_policy.prepare_graphs — the same adapter the
# offline trainer and scripts/multiseed_rl.py use. The hand-written prepare()
# lived here; its hasattr fallbacks are unnecessary on real TNG/CAMELS graphs,
# which always carry stellar_mass/vel_disp/pos.)
from torch_geometric.nn import global_mean_pool
import torch_geometric.data as pg
train_graphs = prepare_graphs(train_loader, gnn, device)
val_graphs = prepare_graphs(val_loader, gnn, device)
test_graphs = prepare_graphs(test_loader, gnn, device)
print("train graphs:", len(train_graphs), "| ctx dim:", train_graphs[0]["ctx"].shape)


In [ ]:
# CELL 7 (retired Sep 2026): ValueNet / compute_advantages / bernoulli_entropy
# now come from rls.policy_gradient (imported in the helpers cell above).
# This cell is intentionally empty — do not re-add inline copies
# (tests/test_notebook_hygiene.py guards against drift).

In [ ]:
# CELL 8 (retired Sep 2026): graph_physics_terms / relative_virial_penalty /
# compute_rewards now come from rls.train_policy and rls.rewards (imported in
# the helpers cell above). CELL 12 below uses those imports directly.
# This cell is intentionally empty — do not re-add inline copies
# (tests/test_notebook_hygiene.py guards against drift).

In [ ]:
# CELL 9 (retired Sep 2026): the GNN adapter was defined here but never used
# by any later cell; eval cells below call the GNN inline (same pattern as
# rls/run_experiment.py). This cell is intentionally empty — do not re-add
# inline copies (tests/test_notebook_hygiene.py guards against drift).

## Load the already-trained policy (instead of retraining)

In [ ]:
# CELL 10-LOAD (replaces re-running B's Cell 10 here): load the policy Notebook B
# already trained instead of training a second one from scratch. This also
# guarantees the numbers below are for the exact policy you're reporting, not a
# fresh, differently-seeded run of it.
#
# Kaggle notebooks are separate sessions — B's /kaggle/working isn't visible here
# automatically. After Notebook B finishes: download rls_outputs.zip (its Cell 16),
# unzip, grab policy.pt, and add it to your "cosmicnet-data" Kaggle Dataset (Dataset
# -> Settings -> New Version -> upload) so it shows up under INPUT below.
import os as _os
rls = cfg["rls"]
# (Sep 2026 rewire: policy class comes from rls.policy — same weights format.)
policy = build_policy(cfg).to(device)
POLICY_INPUT = f"{INPUT}/policy.pt"
# FAIL-CLOSED (audit HIGH-1): the repo ships a STALE PRE-FIX policy.pt at
# outputs/rls/policy.pt — never evaluate it by accident. Upload Notebook B's
# fresh policy.pt to the cosmicnet-data dataset. The override below is for
# debugging ONLY and its metrics are NOT authoritative.
ALLOW_STALE_POLICY_FALLBACK = False
ckpt_path = POLICY_INPUT if _os.path.exists(POLICY_INPUT) else None
if ckpt_path is None and ALLOW_STALE_POLICY_FALLBACK and _os.path.exists("outputs/rls/policy.pt"):
    ckpt_path = "outputs/rls/policy.pt"
    print("=" * 70)
    print("WARNING: ALLOW_STALE_POLICY_FALLBACK=True - evaluating the LOCAL")
    print("outputs/rls/policy.pt (a PRE-FIX smoke artifact committed in the repo).")
    print("Metrics from this run MUST NOT be treated as authoritative.")
    print("=" * 70)
if ckpt_path is None:
    raise RuntimeError(
        "policy.pt not found in the cosmicnet-data dataset. Run Notebook B, "
        "download its policy.pt, upload it to the dataset (Dataset -> Settings "
        "-> New Version -> upload), attach the dataset, and rerun this notebook. "
        "(Set ALLOW_STALE_POLICY_FALLBACK=True only for non-authoritative debugging.)")
policy.load_state_dict(torch.load(ckpt_path, map_location=device))
policy.eval()
print(f"loaded trained policy from {ckpt_path}")
# Integrity: if Notebook B recorded its policy artifact in provenance.json
# (uploaded with the dataset or present locally), verify we are evaluating
# exactly that file. Absent record -> notice only, never a gate.
_prov_path = next((p for p in [f"{INPUT}/provenance.json", "outputs/rls/provenance.json"]
                   if _os.path.exists(p)), None)
_bp = None
if _prov_path:
    try:
        with open(_prov_path) as _f:
            _bp = json.load(_f).get("B_policy")
    except Exception:
        _bp = None
if _bp and _bp.get("policy_artifact", {}).get("md5"):
    _loaded_md5 = _md5_of(ckpt_path)
    if _loaded_md5 != _bp["policy_artifact"]["md5"]:
        raise RuntimeError(
            f"policy.pt integrity mismatch: loaded {_loaded_md5} but Notebook B "
            f"recorded {_bp['policy_artifact']['md5']} - wrong or stale upload.")
    print(f"[policy] integrity OK: matches Notebook B provenance "
          f"(B HEAD={_bp.get('repo', {}).get('head')})")
else:
    print("[policy] no B_policy provenance record found "
          "(older B run or provenance.json not uploaded) - integrity check skipped")


# Decode-mode sync (Sep 2026): eval_mask must use the sparsity_mode the LOADED
# policy was trained under — read it from Notebook B's recorded config, not
# this notebook's inline defaults (which would silently mis-decode a
# topk-trained policy with a 0.5 threshold).
_b_rls = (_bp or {}).get("config_used", {}).get("rls", {}) if isinstance(_bp, dict) else {}
if _b_rls.get("sparsity_mode"):
    rls["sparsity_mode"] = _b_rls["sparsity_mode"]
print(f"[sparsity_mode] eval decoding with sparsity_mode={rls.get('sparsity_mode', 'penalty')!r}" +
      (" (synced from Notebook B provenance)" if _b_rls.get("sparsity_mode") else " (local default — B provenance had no record)"))


## Stage B (optional) — only run this if your Notebook B run also did Stage B

In [ ]:
# CELL 11: Stage B — fine-tune GNN on policy-pruned graphs (rls.stageb.fine_tune_gnn)
# (Sep 2026 rewire: masks via eval_mask; val-guarded fine_tune_gnn instead of
# the fixed 10-epoch loop — best pruned-val epoch without full-val regression
# beyond stageb_full_tol, early-stop, best weights restored.)
# Still gated OFF by default: under Run All this would otherwise silently turn
# every "frozen backbone" number below into a Stage-B number. Flip to True ONLY
# if your Notebook B run also did Stage B.
RUN_STAGE_B = False

ft_graphs = train_graphs[:]
if RUN_STAGE_B:
    _bb_frozen = record_backbone("frozen", gnn)
    masks = []
    with torch.no_grad():
        for g in ft_graphs:
            p = torch.sigmoid(policy(g["edge_attr"].to(device), g["emb"].to(device),
                                     g["edge_index"].to(device), g["ctx"].to(device))).squeeze(-1)
            masks.append(eval_mask(g["edge_index"].to(device), p, rls))
    val_masks = []
    with torch.no_grad():
        for g in val_graphs:
            pv = torch.sigmoid(policy(g["edge_attr"].to(device), g["emb"].to(device),
                                      g["edge_index"].to(device), g["ctx"].to(device))).squeeze(-1)
            val_masks.append(eval_mask(g["edge_index"].to(device), pv, rls))
    history, stageb_info = fine_tune_gnn(
        gnn, ft_graphs, masks, epochs=int(rls.get("stageb_epochs", 10)),
        lr=float(rls.get("stageb_lr", 1e-4)), device=device,
        val_graphs=val_graphs, val_masks=val_masks,
        patience=int(rls.get("stageb_patience", 3)),
        full_tol=float(rls.get("stageb_full_tol", 0.02)))
    print(f"[stageB] best_epoch={stageb_info['best_epoch']} stopped_early={stageb_info['stopped_early']} "
          f"prefinetune_full={stageb_info['prefinetune_full']:.4f} best_full={stageb_info['best_full']} best_pruned={stageb_info['best_pruned']}")
    torch.save(gnn.state_dict(), "outputs/rls/finetuned_gnn.pt")
    print("Stage B done -> outputs/rls/finetuned_gnn.pt")
    record_provenance("C_stageB", extra={
        "backbone_frozen": _bb_frozen,
        "backbone_finetuned": record_backbone("stageB_finetuned", gnn),
    })

else:
    print('Stage B skipped (RUN_STAGE_B=False) - gnn stays frozen for the cells below.')


## CELL B-VERIFY — needed by the results table further down (fix applied, see note above)

In [ ]:
# CELL 12: Verify final policy + fine-tuned GNN on test set
# (Sep 2026 rewire: RL mask via eval_mask — decoding matches the training
# sparsity_mode synced from Notebook B's provenance in CELL 10-LOAD. Also
# retains per-graph masks_rl for the results table in CELL 15.)
from model.physics_loss import MetricsComputer
preds_pol, preds_full, targets, masks_rl = [], [], [], []
with torch.no_grad():
    for b in test_loader:
        b = b.to(device)
        for i in range(b.num_graphs):
            g = b.get_example(i)
            n = g.x.shape[0]
            emb = gnn.get_embeddings(b, embedding_point="pre_pooling")
            ctx = global_mean_pool(emb, b.batch)
            p = torch.sigmoid(policy(g.edge_attr, emb[b.batch == i], g.edge_index, ctx[i])).squeeze(-1)
            m = eval_mask(g.edge_index, p, rls)
            masks_rl.append(m.cpu())
            d = pg.Data(x=g.x, edge_index=g.edge_index[:, m], edge_attr=g.edge_attr[m])
            d.batch = torch.zeros(n, dtype=torch.long, device=device)
            p_pol, _ = gnn(d)
            preds_pol.append(p_pol.item()); targets.append(g.y.item())
            dfull = pg.Data(x=g.x, edge_index=g.edge_index, edge_attr=g.edge_attr)
            dfull.batch = torch.zeros(n, dtype=torch.long, device=device)
            p_full, _ = gnn(dfull)
            preds_full.append(p_full.item())
preds_pol = torch.tensor(preds_pol); preds_full = torch.tensor(preds_full); targets = torch.tensor(targets)
mp, mf = MetricsComputer.compute_all(preds_pol, targets), MetricsComputer.compute_all(preds_full, targets)
print(f"FULL  graph: RMSE={mf['rmse']:.4f} R2={mf['r2']:.4f}")
print(f"RL    pruned: RMSE={mp['rmse']:.4f} R2={mp['r2']:.4f}")
fid = np.corrcoef(preds_pol.numpy(), preds_full.numpy())[0, 1]
print(f"fidelity (Pearson): {fid:.4f}")


## Physics validation, uncertainty calibration, cross-sim, and the final table

In [ ]:
# CELL 12: Physics alignment — binding energy vs keep probability
from scipy.stats import spearmanr
G = 4.302e-9
rhos, pvals, virial_ratios, rel_pens, keeps, dists_kept, dists_dropped = [], [], [], [], [], [], []
with torch.no_grad():
    for b in test_loader:
        b = b.to(device)
        emb = gnn.get_embeddings(b, embedding_point="pre_pooling")
        ctx = global_mean_pool(emb, b.batch)
        for i in range(b.num_graphs):
            g = b.get_example(i)
            p = torch.sigmoid(policy(g.edge_attr, emb[b.batch == i], g.edge_index, ctx[i])).squeeze(-1)
            m = eval_mask(g.edge_index, p, rls)
            u, v = g.edge_index
            stellar = (g.stellar_mass if hasattr(g, "stellar_mass") else torch.ones(g.x.shape[0])*1e10).to(device)
            pos = (g.pos if hasattr(g, "pos") else g.x[:, :3]).to(device)
            r = torch.norm(pos[u] - pos[v], dim=1).clamp(min=1e-6)
            u_ij = G * stellar[u] * stellar[v] / r
            rho, pv = spearmanr(p.cpu().numpy(), u_ij.cpu().numpy())
            rhos.append(rho); pvals.append(pv)
            phys_g = {"stellar_mass": g.stellar_mass if hasattr(g, "stellar_mass") else torch.ones(g.x.shape[0])*1e10,
                      "vel_disp": g.vel_disp if hasattr(g, "vel_disp") else torch.ones(g.x.shape[0])*100,
                      "pos": pos, "edge_index": g.edge_index}
            ke, pe = graph_physics_terms(phys_g, phys_g["edge_index"], m)
            ke_f, pe_f = graph_physics_terms(phys_g, phys_g["edge_index"], torch.ones(g.edge_index.shape[1], dtype=torch.bool, device=device))
            virial_ratios.append(virial_ratio_pruned(ke, pe).item())
            rel_pens.append(relative_virial_penalty(ke, pe, ke_f, pe_f).item())
            keeps.append(m.float().mean().item())
            dists_kept.extend(r[m].cpu().tolist()); dists_dropped.extend(r[~m].cpu().tolist())
print(f"Spearman rho vs U_ij: mean={np.mean(rhos):.3f} (p median={np.median(pvals):.2e})")
print(f"virial ratio (pruned, loop-free): median={np.median(virial_ratios):.3f}")
print(f"relative virial penalty (vs full-graph ref): median={np.median(rel_pens):.3f} IQR=({np.percentile(rel_pens,25):.3f},{np.percentile(rel_pens,75):.3f})")
from scipy.stats import mannwhitneyu
mw = mannwhitneyu(dists_kept, dists_dropped, alternative="less")
print(f"kept edges shorter than dropped? p={mw.pvalue:.2e}")


In [ ]:
# CELL 13: Uncertainty calibration — MC-dropout coverage before/after pruning
def coverage(model, graphs, device, n_samples=30):
    covered, total = 0, 0
    model.eval()
    with torch.no_grad():
        for b in graphs:
            b = b.to(device)
            for i in range(b.num_graphs):
                g = b.get_example(i)
                n = g.x.shape[0]
                d = pg.Data(x=g.x, edge_index=g.edge_index, edge_attr=g.edge_attr)
                d.batch = torch.zeros(n, dtype=torch.long, device=device)
                u = model.predict_with_uncertainty(d, n_samples=n_samples)
                lo, hi = u["mean"] - 1.96*u["std"], u["mean"] + 1.96*u["std"]
                covered += int((g.y >= lo) & (g.y <= hi)); total += 1
    return covered / max(1, total)

print("full-graph 95% CI coverage:", round(coverage(gnn, test_loader, device), 3))
# coverage on pruned graphs requires per-graph Data; reuse Cell 12 masks
print("(pruned coverage computed in same loop as Cell 12 — see results_table)")


In [ ]:
# CELL 14: Cross-simulation OOD — TNG-trained policy on CAMELS
# Requires REAL CAMELS HDF5 (synthetic fallback is NOT publishable — same guard as Notebook D).
import yaml, shutil
cfg2 = yaml.safe_load(open("config/config.yaml"))
cfg2["data"]["source"] = "camels"
cfg2["data"]["camels"] = {"suite": "IllustrisTNG", "simulation": "LH_0",
                          "cache_dir": "/kaggle/working/camels_cache"}
cfg2["data"]["num_workers"] = 0
from data.loaders.base_loader import get_loader
from graph.graph_builder import GraphBuilder
try:
    loader2 = get_loader(cfg2)
    halos2 = loader2.load()
    assert getattr(loader2, "used_synthetic_fallback", False) is False, \
        "synthetic CAMELS fallback is not publishable — cache the real HDF5 first"
    print("CAMELS halos loaded:", len(halos2))
    gb2 = GraphBuilder(cfg2)
    graphs2 = gb2.build_graphs(halos2[:100])
    predsF, predsP, ys = [], [], []
    from torch_geometric.data import Batch as PyGBatch
    with torch.no_grad():
        for g in graphs2:
            g = g.to(device)
            # get_embeddings/forward REQUIRE a Batch (has .batch); a bare Data crashes
            gb = PyGBatch.from_data_list([g])
            emb = gnn.get_embeddings(gb, embedding_point="pre_pooling")
            ctx = global_mean_pool(emb, gb.batch)   # [1, out] on a single-graph batch
            # policy expects a 1-D context: [1, out] would unsqueeze to [1, 1, out]
            # and crash in expand() (caught by the except below and misread as a
            # missing-HDF5 error). Same convention as rls/cross_sim.py and Notebook D.
            p = torch.sigmoid(policy(g.edge_attr, emb, g.edge_index, ctx[0])).squeeze(-1)
            m = eval_mask(g.edge_index, p, rls)
            d = pg.Data(x=g.x, edge_index=g.edge_index, edge_attr=g.edge_attr)
            d.batch = torch.zeros(g.x.shape[0], dtype=torch.long, device=device)
            pf, _ = gnn(d)
            predsF.append(pf.item())
            dp = pg.Data(x=g.x, edge_index=g.edge_index[:, m], edge_attr=g.edge_attr[m])
            dp.batch = torch.zeros(g.x.shape[0], dtype=torch.long, device=device)
            pp, _ = gnn(dp)
            predsP.append(pp.item())
            ys.append(g.y.item())
    mf2 = MetricsComputer.compute_all(torch.tensor(predsF), torch.tensor(ys))
    mp2 = MetricsComputer.compute_all(torch.tensor(predsP), torch.tensor(ys))
    print(f"CROSS-SIM (TNG->CAMELS): full RMSE={mf2['rmse']:.4f} | RL-pruned RMSE={mp2['rmse']:.4f}")
    pd.DataFrame({"y": ys, "pred_full": predsF, "pred_pruned": predsP}).to_csv(
        "outputs/rls/cross_sim_results.csv", index=False)
except Exception as e:
    print("cross-sim failed (likely no HDF5 / no internet to Flatiron):", e)
    print("Synthetic CAMELS fallback was used by the loader; check the loader's _generate_synthetic_camels path.")


In [ ]:
# CELL 15: Final results table + paper plots — aggregation via rls.evaluate
# (Sep 2026 rewire: build_results_table replaces the hand-rolled means, so the
# table cannot drift from scripts/multiseed_rl.py again. Per-method preds/masks
# are still computed in the cells above — CELL 12 now also retains masks_rl.
# baselines.csv rows carry no raw preds/masks, so they merge afterwards exactly
# as before.)
rows = build_results_table(preds_full, preds_pol, None, None, targets,
                           masks_rl, None, None)
# Notebook A wrote baselines.csv in ITS session; Kaggle sessions don't share
# /kaggle/working, so prefer the copy uploaded to the cosmicnet-data dataset
# (this file is NOT committed to the repo -> the bare path below only works if
# Notebook A ran in THIS session).
bl_candidates = [f"{INPUT}/baselines.csv", "outputs/rls/baselines.csv"]
bl_path = next((p for p in bl_candidates if os.path.exists(p)), None)
assert bl_path is not None, (
    "baselines.csv not found — run Notebook A, download its rls_outputs.zip, "
    "and add baselines.csv to the cosmicnet-data Kaggle dataset (next to policy.pt)")
bl = pd.read_csv(bl_path)
# Freshness diagnostic only (audit MEDIUM-1) - does NOT change selection logic
# or values; makes a stale uploaded baselines.csv diagnosable.
import datetime as _dt, stat as _stat
_st = os.stat(bl_path)
print(f"[baselines] using {bl_path}: exists=True, {_st.st_size} bytes, "
      f"mtime {_dt.datetime.fromtimestamp(_st.st_mtime)}")
_ap = None
for _p in [f"{INPUT}/provenance.json", "outputs/rls/provenance.json"]:
    if os.path.exists(_p):
        try:
            with open(_p) as _f:
                _ap = json.load(_f).get("A")
        except Exception:
            pass
        break
if _ap:
    print(f"[baselines] Notebook A provenance: HEAD={_ap.get('repo', {}).get('head')}, "
          f"csv md5={_ap.get('dataset', {}).get('tng_csv', {}).get('md5')}")
else:
    print("[baselines] no Notebook A provenance available (older A run or "
          "provenance.json not uploaded) - freshness cannot be confirmed")
for _, r in bl.iterrows():
    rows.append({"method": r.method, "rmse": r.rmse, "r2": r.r2,
                 "scatter": float("nan"), "keep_frac": r.keep_frac, "fidelity": float("nan")})
# Backbone provenance guard (Sep 2026): every reported row says which backbone
# produced it — the B-vs-C silent swap (R2 0.9075 vs 0.8159) cannot recur.
_bb_stage = "stageB_finetuned" if RUN_STAGE_B else "frozen"
_bb_rec = record_backbone(_bb_stage, gnn)
require_backbone_label(rows, _bb_rec)
print(f"[backbone] {format_backbone_label(_bb_rec)}")
res = pd.DataFrame(rows)
save_paper_plots(rows, out_dir="outputs/rls")  # canonical results_table.csv + pareto_and_fidelity.png
print(res.to_string(index=False))

# Canonical Pareto + fidelity panels come from save_paper_plots above; this
# notebook keeps its extra pred-vs-true panel here.
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
# FIX: previously hardcoded to 5 methods and silently dropped mass_ratio, grad_saliency,
# attention_topk, and gumbel from the plot even though they were computed. Now plots
# every method present in the results table.
plot_order = ["full", "random", "degree", "distance", "mass_ratio", "attention_topk",
              "grad_saliency", "gumbel", "rl_policy"]
for name in plot_order:
    sub = res[res.method == name]
    if not sub.empty:
        axes[0].plot(sub.keep_frac, sub.rmse, "o-", label=name)
axes[0].set_xlabel("mean keep fraction"); axes[0].set_ylabel("RMSE (dex)")
axes[0].set_xlim(1.05, -0.05); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].scatter(targets, preds_full, s=15, alpha=0.5, label="full")
axes[1].scatter(targets, preds_pol, s=15, alpha=0.5, marker="x", label="rl-pruned")
axes[1].plot([targets.min(), targets.max()], [targets.min(), targets.max()], "k--")
axes[1].set_xlabel("true log M_halo"); axes[1].set_ylabel("predicted"); axes[1].legend()
fig.tight_layout(); fig.savefig("outputs/rls/paper_figures.png", dpi=200)
print("saved outputs/rls/results_table.csv + paper_figures.png")


In [ ]:
# CELL 16: Save everything to Kaggle output for download
shutil.make_archive("/kaggle/working/rls_outputs", "zip", "outputs/rls")
print("Download /kaggle/working/rls_outputs.zip — contains all results, plots, policy.pt, finetuned_gnn.pt")
